# Module 07: Monte Carlo Analysis

Monte Carlo (MC) analysis runs the same simulation many times with varied parameters to characterize statistical performance — essential for flight software validation.

### Learning Objectives
- Use Basilisk's built-in `MonteCarloController`
- Define parameter dispersions (Gaussian, uniform, initial conditions)
- Run parallel MC batches
- Aggregate and visualize statistical results
- Compute 3σ performance bounds

---

## Why Monte Carlo?

| Question | MC Answer |
|---|---|
| Does the controller always converge? | Run 1000 cases with random ICs |
| What's the worst-case settling time? | Look at 99th percentile |
| How does sensor noise affect accuracy? | Vary noise seed across runs |
| Do inertia errors destabilize control? | Disperse mass properties |

---

## Basilisk MC Framework

BSK's `MonteCarloController` manages:
1. A **scenario function** (your simulation setup)
2. **Dispersions** (what parameters vary and how)
3. **Retention policies** (what data to keep)
4. Parallel execution via Python `multiprocessing`

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('plots', exist_ok=True)
os.makedirs('mc_results', exist_ok=True)

from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
from Basilisk.utilities import RigidBodyKinematics as rbk
from Basilisk.utilities import simIncludeGravBody, simIncludeRW
from Basilisk.simulation import spacecraft, reactionWheelStateEffector, simpleNav
from Basilisk.fswAlgorithms import inertial3D, attTrackingError, mrpFeedback, rwMotorTorque
from Basilisk.fswAlgorithms import vehicleConfigData
from Basilisk.architecture import bskLogging

# Basilisk MC controller
from Basilisk.utilities.MonteCarlo import Controller as MC
from Basilisk.utilities.MonteCarlo import RetentionPolicy
from Basilisk.utilities.MonteCarlo import Dispersions as bskDisp  # capital D in BSK 2.9+

bskLogging.setDefaultLogLevel(bskLogging.BSK_ERROR)  # quiet for MC runs
print("Imports OK.")

Imports OK.


---

## Step 1 — Define the Scenario Function

The scenario function must accept a `SimBaseClass` and a dictionary of scenario parameters, then set up and return the simulation.

In [2]:
def scenario_attitude_control(show_plots=False):
    """
    Baseline attitude control scenario for MC analysis.
    Returns the simulation object and data recorders.
    """
    from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
    from Basilisk.utilities import simIncludeGravBody, simIncludeRW
    from Basilisk.simulation import spacecraft, reactionWheelStateEffector, simpleNav
    from Basilisk.fswAlgorithms import inertial3D, attTrackingError, mrpFeedback, rwMotorTorque
    from Basilisk.fswAlgorithms import vehicleConfigData

    mu_earth = 3.986004418e14
    R_earth  = 6371e3

    scSim = SimulationBaseClass.SimBaseClass()
    dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
    fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)

    dynTask = scSim.CreateNewTask("DynamicsTask", macros.sec2nano(0.5))
    fswTask = scSim.CreateNewTask("FswTask",      macros.sec2nano(1.0))
    dynProcess.addTask(dynTask)
    fswProcess.addTask(fswTask)

    # Spacecraft
    scObject = spacecraft.Spacecraft()
    scObject.ModelTag = "BSKsat"
    Ixx, Iyy, Izz = 0.025, 0.050, 0.065
    I3x3 = np.diag([Ixx, Iyy, Izz])
    scObject.hub.mHub = 10.0
    scObject.hub.IHubPntBc_B = unitTestSupport.np2EigenMatrix3d(I3x3.flatten())
    scObject.hub.r_BcB_B = [[0.0], [0.0], [0.0]]

    oe = orbitalMotion.ClassicElements()
    oe.a = R_earth + 500e3
    oe.e = oe.i = oe.Omega = oe.omega = oe.f = 0.0
    r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)
    scObject.hub.r_CN_NInit = r0.tolist()
    scObject.hub.v_CN_NInit = v0.tolist()
    scObject.hub.sigma_BNInit   = [[0.2], [0.1], [-0.15]]
    scObject.hub.omega_BN_BInit = [[0.01], [-0.01], [0.005]]

    gravFactory = simIncludeGravBody.gravBodyFactory()
    earth = gravFactory.createEarth()
    earth.isCentralBody = True
    gravFactory.addBodiesTo(scObject)
    scSim.AddModelToTask("DynamicsTask", scObject)

    # RWs
    rwFactory = simIncludeRW.rwFactory()
    varRWModel = reactionWheelStateEffector.BalancedWheels
    for axis in [[1,0,0],[0,1,0],[0,0,1]]:
        rwFactory.create('Honeywell_HR16', gsHat_B=axis,
                         maxMomentum=50.0, Omega=0.0, RWModel=varRWModel)
    rwStateEffector = reactionWheelStateEffector.ReactionWheelStateEffector()
    rwStateEffector.ModelTag = "RWArray"
    rwFactory.addToSpacecraft(scObject.ModelTag, rwStateEffector, scObject)
    scSim.AddModelToTask("DynamicsTask", rwStateEffector)

    # SimpleNav: converts scStateOutMsg → attOutMsg / transOutMsg
    sNavObj = simpleNav.SimpleNav()
    sNavObj.ModelTag = "SimpleNav"
    sNavObj.scStateInMsg.subscribeTo(scObject.scStateOutMsg)
    scSim.AddModelToTask("DynamicsTask", sNavObj)

    # FSW
    inertial3DObj = inertial3D.inertial3D()
    inertial3DObj.ModelTag = "inertial3D"
    inertial3DObj.sigma_R0N = [0.0, 0.0, 0.0]
    scSim.AddModelToTask("FswTask", inertial3DObj)

    attErrObj = attTrackingError.attTrackingError()
    attErrObj.ModelTag = "attTrackingError"
    attErrObj.attNavInMsg.subscribeTo(sNavObj.attOutMsg)
    attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)
    scSim.AddModelToTask("FswTask", attErrObj)

    K = 2.0 * 0.7 * 0.15 * (Ixx + Iyy + Izz)
    P = 0.15**2 * (Ixx + Iyy + Izz)
    mrpControlObj = mrpFeedback.mrpFeedback()
    mrpControlObj.ModelTag = "MRP_Feedback"
    mrpControlObj.K  = K
    mrpControlObj.P  = P
    mrpControlObj.Ki = -1.0
    mrpControlObj.integralLimit = 2.0

    vehConfigObj = vehicleConfigData.vehicleConfigData()
    vehConfigObj.ModelTag = "vehicleConfigData"
    vehConfigObj.ISCPntB_B = I3x3.flatten().tolist()
    scSim.AddModelToTask("FswTask", vehConfigObj)

    mrpControlObj.guidInMsg.subscribeTo(attErrObj.attGuidOutMsg)
    mrpControlObj.vehConfigInMsg.subscribeTo(vehConfigObj.vecConfigOutMsg)
    mrpControlObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
    mrpControlObj.rwSpeedsInMsg.subscribeTo(rwStateEffector.rwSpeedOutMsg)
    scSim.AddModelToTask("FswTask", mrpControlObj)

    rwMotorTorqueObj = rwMotorTorque.rwMotorTorque()
    rwMotorTorqueObj.ModelTag = "rwMotorTorque"
    rwMotorTorqueObj.controlAxes_B = [1,0,0, 0,1,0, 0,0,1]
    rwMotorTorqueObj.vehControlInMsg.subscribeTo(mrpControlObj.cmdTorqueOutMsg)
    rwMotorTorqueObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
    rwStateEffector.rwMotorCmdInMsg.subscribeTo(rwMotorTorqueObj.rwMotorTorqueOutMsg)
    scSim.AddModelToTask("FswTask", rwMotorTorqueObj)

    # Recorders
    attErrRec = attErrObj.attGuidOutMsg.recorder()
    scSim.AddModelToTask("FswTask", attErrRec)

    scSim.InitializeSimulation()
    scSim.ConfigureStopTime(macros.sec2nano(300.0))
    scSim.ExecuteSimulation()

    return scSim, attErrRec


# Test the baseline scenario
sim_test, rec_test = scenario_attitude_control()
t_test = rec_test.times() * macros.NANO2SEC
err_test = np.degrees(4 * np.arctan(np.linalg.norm(rec_test.sigma_BR, axis=1)))
print(f"Baseline: initial error = {err_test[0]:.2f} deg, final error = {err_test[-1]:.4f} deg")

[FILE : /project/dist3/autoSource/cMsgCInterface/RWArrayConfigMsg_C.cpp, FUNC : RWArrayConfigMsg_C_read, LINE : 79]:
MSG_ERROR: In C input msg, you are trying to read an un-written message of type RWArrayConfigMsg.
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE : 2075]:
MSG_ERROR: Error: singular 3x3 matrix inverse
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE : 2075]:
MSG_ERROR: Error: singular 3x3 matrix inverse
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE : 2075]:
MSG_ERROR: Error: singular 3x3 matrix inverse
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE : 2075]:
MSG_ERROR: Error: singular 3x3 matrix inverse
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE : 2075]:
MSG_ERROR: Error: singular 3x3 matrix inverse
[FILE : /project/src/architecture/utilities/linearAlgebra.c, FUNC : m33Inverse, LINE 

---

## Step 2 — Manual Monte Carlo (Pure Python Approach)

For tight integration with notebooks, we implement a straightforward MC loop with parameter dispersions.

In [3]:
def run_mc_scenario(sigma0_disp, omega0_disp):
    """
    Run one MC case with dispersed initial conditions.
    Returns: (settling_time_s, final_error_deg, t_array, error_array)
    """
    from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
    from Basilisk.utilities import simIncludeGravBody, simIncludeRW
    from Basilisk.simulation import spacecraft, reactionWheelStateEffector, simpleNav
    from Basilisk.fswAlgorithms import inertial3D, attTrackingError, mrpFeedback, rwMotorTorque
    from Basilisk.fswAlgorithms import vehicleConfigData
    from Basilisk.architecture import bskLogging
    bskLogging.setDefaultLogLevel(bskLogging.BSK_ERROR)

    mu_earth = 3.986004418e14
    R_earth  = 6371e3

    scSim = SimulationBaseClass.SimBaseClass()
    dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
    fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)
    dynTask = scSim.CreateNewTask("DynamicsTask", macros.sec2nano(0.5))
    fswTask = scSim.CreateNewTask("FswTask",      macros.sec2nano(1.0))
    dynProcess.addTask(dynTask)
    fswProcess.addTask(fswTask)

    scObject = spacecraft.Spacecraft()
    scObject.ModelTag = "BSKsat"
    Ixx, Iyy, Izz = 0.025, 0.050, 0.065
    I3x3 = np.diag([Ixx, Iyy, Izz])
    scObject.hub.mHub = 10.0
    scObject.hub.IHubPntBc_B = unitTestSupport.np2EigenMatrix3d(I3x3.flatten())
    scObject.hub.r_BcB_B = [[0.0], [0.0], [0.0]]

    oe = orbitalMotion.ClassicElements()
    oe.a = R_earth + 500e3
    oe.e = oe.i = oe.Omega = oe.omega = oe.f = 0.0
    r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)
    scObject.hub.r_CN_NInit = r0.tolist()
    scObject.hub.v_CN_NInit = v0.tolist()
    scObject.hub.sigma_BNInit   = [[sigma0_disp[0]], [sigma0_disp[1]], [sigma0_disp[2]]]
    scObject.hub.omega_BN_BInit = [[omega0_disp[0]], [omega0_disp[1]], [omega0_disp[2]]]

    gravFactory = simIncludeGravBody.gravBodyFactory()
    earth = gravFactory.createEarth()
    earth.isCentralBody = True
    gravFactory.addBodiesTo(scObject)
    scSim.AddModelToTask("DynamicsTask", scObject)

    rwFactory = simIncludeRW.rwFactory()
    varRWModel = reactionWheelStateEffector.BalancedWheels
    for axis in [[1,0,0],[0,1,0],[0,0,1]]:
        rwFactory.create('Honeywell_HR16', gsHat_B=axis,
                         maxMomentum=50.0, Omega=0.0, RWModel=varRWModel)
    rwStateEffector = reactionWheelStateEffector.ReactionWheelStateEffector()
    rwStateEffector.ModelTag = "RWArray"
    rwFactory.addToSpacecraft(scObject.ModelTag, rwStateEffector, scObject)
    scSim.AddModelToTask("DynamicsTask", rwStateEffector)

    sNavObj = simpleNav.SimpleNav()
    sNavObj.ModelTag = "SimpleNav"
    sNavObj.scStateInMsg.subscribeTo(scObject.scStateOutMsg)
    scSim.AddModelToTask("DynamicsTask", sNavObj)

    inertial3DObj = inertial3D.inertial3D()
    inertial3DObj.ModelTag = "inertial3D"
    inertial3DObj.sigma_R0N = [0.0, 0.0, 0.0]
    scSim.AddModelToTask("FswTask", inertial3DObj)

    attErrObj = attTrackingError.attTrackingError()
    attErrObj.ModelTag = "attTrackingError"
    attErrObj.attNavInMsg.subscribeTo(sNavObj.attOutMsg)
    attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)
    scSim.AddModelToTask("FswTask", attErrObj)

    K = 2.0 * 0.7 * 0.15 * (Ixx + Iyy + Izz)
    P = 0.15**2 * (Ixx + Iyy + Izz)
    mrpControlObj = mrpFeedback.mrpFeedback()
    mrpControlObj.ModelTag = "MRP_Feedback"
    mrpControlObj.K  = K
    mrpControlObj.P  = P
    mrpControlObj.Ki = -1.0
    mrpControlObj.integralLimit = 2.0

    vehConfigObj = vehicleConfigData.vehicleConfigData()
    vehConfigObj.ModelTag = "vehicleConfigData"
    vehConfigObj.ISCPntB_B = I3x3.flatten().tolist()
    scSim.AddModelToTask("FswTask", vehConfigObj)

    mrpControlObj.guidInMsg.subscribeTo(attErrObj.attGuidOutMsg)
    mrpControlObj.vehConfigInMsg.subscribeTo(vehConfigObj.vecConfigOutMsg)
    mrpControlObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
    mrpControlObj.rwSpeedsInMsg.subscribeTo(rwStateEffector.rwSpeedOutMsg)
    scSim.AddModelToTask("FswTask", mrpControlObj)

    rwMotorTorqueObj = rwMotorTorque.rwMotorTorque()
    rwMotorTorqueObj.ModelTag = "rwMotorTorque"
    rwMotorTorqueObj.controlAxes_B = [1,0,0, 0,1,0, 0,0,1]
    rwMotorTorqueObj.vehControlInMsg.subscribeTo(mrpControlObj.cmdTorqueOutMsg)
    rwMotorTorqueObj.rwParamsInMsg.subscribeTo(rwFactory.getConfigMessage())
    rwStateEffector.rwMotorCmdInMsg.subscribeTo(rwMotorTorqueObj.rwMotorTorqueOutMsg)
    scSim.AddModelToTask("FswTask", rwMotorTorqueObj)

    attErrRec = attErrObj.attGuidOutMsg.recorder()
    scSim.AddModelToTask("FswTask", attErrRec)

    scSim.InitializeSimulation()
    scSim.ConfigureStopTime(macros.sec2nano(300.0))
    scSim.ExecuteSimulation()

    t_arr   = attErrRec.times() * macros.NANO2SEC
    err_arr = np.degrees(4 * np.arctan(np.linalg.norm(attErrRec.sigma_BR, axis=1)))

    # Find settling time (first time error < 1 deg and stays there)
    settled = np.where(err_arr < 1.0)[0]
    t_settle = t_arr[settled[0]] if len(settled) > 0 else np.nan

    return t_settle, err_arr[-1], t_arr, err_arr


print("MC scenario function defined.")

MC scenario function defined.


In [ ]:
# ── Run Monte Carlo — 50 cases with dispersed initial conditions ──────────────
np.random.seed(42)
N_MC = 50

# Dispersions: initial MRP within ±0.4 (roughly ±80°), angular rate ±0.03 rad/s
sigma0_mean  = np.array([0.0,  0.0, 0.0])
sigma0_std   = 0.2
omega0_mean  = np.array([0.0, 0.0, 0.0])
omega0_std   = 0.015  # rad/s

mc_results = []

print(f"Running {N_MC} Monte Carlo cases...")
for i in range(N_MC):
    sigma_ic = sigma0_mean + np.random.normal(0, sigma0_std, 3)
    omega_ic = omega0_mean + np.random.normal(0, omega0_std, 3)

    t_settle, final_err, t_arr, err_arr = run_mc_scenario(sigma_ic, omega_ic)
    mc_results.append({
        'sigma_ic': sigma_ic,
        'omega_ic': omega_ic,
        't_settle': t_settle,
        'final_err': final_err,
        't': t_arr,
        'err': err_arr,
    })

    if (i+1) % 10 == 0:
        print(f"  Completed {i+1}/{N_MC}")

print("MC run complete.")

In [ ]:
# ── Extract statistics ────────────────────────────────────────────────────────
t_settle_all  = np.array([r['t_settle']  for r in mc_results])
final_err_all = np.array([r['final_err'] for r in mc_results])

settled_cases = np.sum(~np.isnan(t_settle_all))

print(f"Cases that settled (< 1 deg): {settled_cases}/{N_MC} ({100*settled_cases/N_MC:.0f}%)")
print(f"\nSettling time (of settled cases):")
t_s = t_settle_all[~np.isnan(t_settle_all)]
print(f"  Mean:   {t_s.mean():.1f} s")
print(f"  Std:    {t_s.std():.1f} s")
print(f"  50th %: {np.percentile(t_s, 50):.1f} s")
print(f"  95th %: {np.percentile(t_s, 95):.1f} s")
print(f"  99th %: {np.percentile(t_s, 99):.1f} s")

print(f"\nFinal attitude error (all cases):")
print(f"  Mean:   {final_err_all.mean():.4f} deg")
print(f"  3-sigma: {3*final_err_all.std():.4f} deg")
print(f"  Max:    {final_err_all.max():.4f} deg")

In [ ]:
# ── Plot 1: All MC error trajectories ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for r in mc_results:
    ax.plot(r['t'], r['err'], color='steelblue', alpha=0.2, linewidth=0.8)

# Compute percentile envelopes
t_common = mc_results[0]['t']
err_matrix = np.array([r['err'] for r in mc_results])
p50 = np.percentile(err_matrix, 50, axis=0)
p95 = np.percentile(err_matrix, 95, axis=0)
p99 = np.percentile(err_matrix, 99, axis=0)

ax.plot(t_common, p50, 'k-',   linewidth=2, label='50th percentile')
ax.plot(t_common, p95, 'r--',  linewidth=2, label='95th percentile')
ax.plot(t_common, p99, 'r:',   linewidth=2, label='99th percentile')
ax.axhline(1.0, color='orange', linestyle='--', label='1 deg threshold')

ax.set_yscale('log')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Attitude Error (deg)')
ax.set_title(f'Monte Carlo Attitude Error — {N_MC} Cases')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/07_mc_error_trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

# ── Plot 2: Settling time histogram ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(t_s, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(np.percentile(t_s, 95), color='red', linestyle='--',
           label=f'95th %ile = {np.percentile(t_s, 95):.0f} s')
ax.axvline(t_s.mean(), color='orange', linestyle='-',
           label=f'Mean = {t_s.mean():.0f} s')
ax.set_xlabel('Settling Time (s)')
ax.set_ylabel('Count')
ax.set_title('Monte Carlo Settling Time Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/07_mc_settling_histogram.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: Initial error magnitude vs settling time scatter ──────────────────
ic_err = np.array([np.degrees(4*np.arctan(np.linalg.norm(r['sigma_ic']))) for r in mc_results])

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(ic_err, t_settle_all, c=final_err_all, cmap='viridis',
                     s=60, alpha=0.8)
plt.colorbar(scatter, ax=ax, label='Final Error (deg)')
ax.set_xlabel('Initial Attitude Error (deg)')
ax.set_ylabel('Settling Time (s)')
ax.set_title('MC: Initial Error vs Settling Time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/07_mc_ic_vs_settling.png', dpi=100, bbox_inches='tight')
plt.show()

---

## Using Basilisk's Built-in MC Controller

For larger, production MC campaigns, use `MonteCarloController`:

In [ ]:
# ── Basilisk MonteCarloController pattern (pseudocode illustration) ───────────
print("""
# ── Basilisk MC Controller (production pattern) ──────────────────────────────
#
# from Basilisk.utilities.MonteCarlo import Controller, RetentionPolicy
# from Basilisk.utilities.MonteCarlo import dispersions as bskDisp
#
# mc = Controller.MonteCarlo()
# mc.setSimulationFunction(my_scenario_function)   # function that builds the sim
# mc.setExecutionFunction(my_exec_function)         # function that runs the sim
# mc.setExecutionCount(500)                         # number of MC runs
# mc.archiveDir = 'mc_archive/'                     # where to save results
# mc.setShouldDisperseSeeds(True)                   # randomize RNG seeds
#
# # Add dispersions
# disp_sigma = bskDisp.InertiaTensorDispersion('hub.IHubPntBc_B', stdDeviation=0.05)
# mc.addDispersion(disp_sigma)
#
# disp_ic = bskDisp.NormalVectorCartDispersion('hub.r_CN_NInit', 0.0, [10.0, 10.0, 10.0])
# mc.addDispersion(disp_ic)
#
# # Add retention policy (what data to save)
# retainPolicy = RetentionPolicy.RetentionPolicy()
# retainPolicy.addMessageLog("attGuidOutMsg", ["sigma_BR"])
# mc.addRetentionPolicy(retainPolicy)
#
# # Run (parallel by default)
# mc.execute()
""")

---

## Summary

- Monte Carlo analysis validates controller **robustness** across a population of scenarios
- Disperse: initial conditions, inertia properties, sensor noise seeds, orbital parameters
- Key metrics: **settling time CDF**, **final error statistics**, **failure rate**
- Basilisk's `MonteCarloController` supports parallel execution and automatic archiving
- For notebooks, a manual loop (as shown) is simple and fully transparent

**Next: [08 - Custom Module Development](08_custom_modules.ipynb)**